# Policy iteration

_Reinforcement Learning_

**Score a plan, then make it greedy; repeat until the plan stops changing.**

We want the best plan for a known world. Policy iteration finds it by alternating two moves until they agree.

       (1) Evaluate. Take the current plan $\pi$ and compute how good every state is if you follow $\pi$ forever. Call this score $V^\pi(s)$.

---

This notebook is a practice scaffold for the **Policy iteration** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

In [ ]:
import numpy as np

# 4x4 gridworld. States 0..15 row-major. Corners 0 and 15 are terminal.
# Every move costs -1; gamma = 1. Actions: 0=up, 1=down, 2=left, 3=right.
N, A = 4, 4
TERMINAL = {0, 15}
MOVE = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
ARROW = {0: "up", 1: "down", 2: "left", 3: "right"}
gamma = 1.0

def next_state(s, a):
    if s in TERMINAL:
        return s
    r, c = divmod(s, N)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if 0 <= nr < N and 0 <= nc < N:      # stay put if the move hits a wall
        return nr * N + nc
    return s

def policy_eval(policy, tol=1e-4, max_sweeps=1000):
    """Iterative policy evaluation: apply the Bellman EXPECTATION backup
       V(s) <- R + gamma * V(s') until the largest change drops below tol."""
    V = np.zeros(N * N)
    for _ in range(max_sweeps):
        delta = 0.0
        for s in range(N * N):
            if s in TERMINAL:
                continue
            s2 = next_state(s, policy[s])
            v_new = -1.0 + gamma * V[s2]   # deterministic move, reward -1
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        if delta < tol:
            break
    return V

def greedy_improve(V):
    """Policy IMPROVEMENT: pi'(s) = argmax_a [ R + gamma V(s') ].
       Ties broken by lowest action index (np.argmax) so the loop terminates."""
    policy = np.zeros(N * N, dtype=int)
    for s in range(N * N):
        if s in TERMINAL:
            continue
        q = [-1.0 + gamma * V[next_state(s, a)] for a in range(A)]
        policy[s] = int(np.argmax(q))
    return policy

# Start from an arbitrary policy (always 'up') and alternate the two steps.
policy = np.zeros(N * N, dtype=int)
for step in range(50):
    V = policy_eval(policy)                # (1) evaluate current policy
    new_policy = greedy_improve(V)         # (2) improve greedily
    changed = int(np.sum(new_policy != policy))
    print(f"improvement step {step}: {changed} states changed action")
    if changed == 0:                       # policy stable => optimal
        break
    policy = new_policy

print("\noptimal value V* (4x4 grid):")
print(np.round(V.reshape(N, N), 1))
print("\noptimal policy pi* (4x4 grid):")
grid = np.array([("term" if s in TERMINAL else ARROW[policy[s]])
                 for s in range(N * N)]).reshape(N, N)
print(grid)

## Visualize the data & results

_On the classic 4x4 gridworld, how fast does policy iteration lock in the optimal policy?_

In [ ]:
import numpy as np

# 4x4 gridworld: corners 0 and 15 terminal, every move costs -1, gamma = 1.
N, A = 4, 4
TERMINAL = {0, 15}
MOVE = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}   # up, down, left, right
gamma = 1.0

def next_state(s, a):
    if s in TERMINAL:
        return s
    r, c = divmod(s, N)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    return nr * N + nc if (0 <= nr < N and 0 <= nc < N) else s

def policy_eval(policy, tol=1e-4, max_sweeps=1000):
    V = np.zeros(N * N)
    for _ in range(max_sweeps):
        delta = 0.0
        for s in range(N * N):
            if s in TERMINAL:
                continue
            v = -1.0 + gamma * V[next_state(s, policy[s])]
            delta = max(delta, abs(v - V[s]))
            V[s] = v
        if delta < tol:
            break
    return V

policy = np.zeros(N * N, dtype=int)       # arbitrary start: 'always up'
changes = []
for _ in range(50):
    V = policy_eval(policy)
    new_policy = np.array([
        int(np.argmax([-1.0 + gamma * V[next_state(s, a)] for a in range(A)]))
        if s not in TERMINAL else 0
        for s in range(N * N)
    ])
    changed = int(np.sum(new_policy != policy))
    changes.append(changed)
    if changed == 0:
        break
    policy = new_policy

print("states changed per improvement step:", changes)  # [6, 7, 2, 0]
print("optimal V*:\n", np.round(V.reshape(N, N), 1))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Under policy $\pi$, state $s$ has $V^\pi(s) = 4$. A one-step look-ahead gives action-values $Q^\pi(s,\text{up}) = 4$, $Q^\pi(s,\text{down}) = 7$, $Q^\pi(s,\text{stay}) = 1$. What does the improvement step set $\pi'(s)$ to, and what does the policy-improvement theorem promise?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute $\arg\max_a Q^\pi(s,a) = \arg\max(4,7,1)$. — _The improvement step picks the action with the highest one-step look-ahead value._
- Compare the best $Q$, which is $7$, against $V^\pi(s) = 4$. — _The theorem's premise is $Q^\pi(s,\pi'(s)) \ge V^\pi(s)$; here $7 \ge 4$._

**Answer:** $\pi'(s) = \text{down}$ (the $\arg\max$). Since $Q^\pi(s,\text{down}) = 7 \ge 4 = V^\pi(s)$, the policy-improvement theorem guarantees $V^{\pi'} \ge V^\pi$ at every state &mdash; the new plan is no worse anywhere, and strictly better at $s$.

</details>

**Problem 2.** You run an improvement step and not a single state's action changes. What do you conclude, and why is it safe to stop?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that "greedy with respect to $V^\pi$ equals $\pi$ itself". — _No action changed means the $\arg\max$ already agreed with the current policy in every state._
- Write down what that fixed point says: $V^\pi(s) = \max_a \sum_{s'} P\,[R + \gamma V^\pi(s')]$. — _Greedy-equals-current means the value already satisfies the Bellman OPTIMALITY equation, not just the expectation equation._

**Answer:** The policy is stable: it already satisfies the Bellman optimality equation, whose only solution is $V^*$. So the current policy is $\pi^*$ and you stop. (This is also why ties must be broken deterministically &mdash; otherwise the $\arg\max$ could "change" between equal-value actions and the stable condition would never register.)

</details>

**Problem 3.** Policy iteration evaluates each policy to full convergence before improving. Name a cheaper variant and the extreme case where it becomes value iteration.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that the greedy improvement only needs an APPROXIMATE $V^\pi$ to point the right way. — _Exact evaluation is wasteful (a listed pitfall); a few sweeps usually suffice to get the $\arg\max$ right._
- Consider truncating evaluation to a fixed number $k$ of sweeps, then to $k=1$. — _Truncated evaluation interpolates between policy iteration ($k=\infty$) and value iteration ($k=1$)._

**Answer:** Modified policy iteration: run only $k$ evaluation sweeps before each improvement instead of iterating to convergence. As $k \to 1$ &mdash; one evaluation sweep then an improvement &mdash; the combined update becomes $V(s) \leftarrow \max_a \sum_{s'} P\,[R + \gamma V(s')]$, which is exactly the value-iteration backup. So value iteration is the extreme, fully-truncated case of policy iteration.

</details>